In [1]:
import pandas as pd

df = pd.read_csv('/content/multimedia_knowledge_dataset.csv')

display(df.head())

,text,topic,source_type,source_credibility,article_length,contradiction_score,semantic_similarity,readability_score,sentiment,trust_score
0,Politics related content discussing recent dev...,Politics,Government Website,8,3858,0.66,0.75,78.72,Neutral,74.54
1,Education related content discussing recent de...,Education,Research Paper,9,2895,0.87,0.56,89.87,Positive,83.81
2,Finance related content discussing recent deve...,Finance,Government Website,8,166,0.34,0.50,83.86,Positive,85.50
3,Finance related content discussing recent deve...,Finance,Government Website,8,4729,0.57,0.48,88.89,Positive,79.80
4,Healthcare related content discussing recent d...,Healthcare,News Article,6,2856,0.88,0.73,59.13,Neutral,54.79


In [2]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   text                 1500 non-null   object 
 1   topic                1500 non-null   object 
 2   source_type          1500 non-null   object 
 3   source_credibility   1500 non-null   int64  
 4   article_length       1500 non-null   int64  
 5   contradiction_score  1500 non-null   float64
 6   semantic_similarity  1500 non-null   float64
 7   readability_score    1500 non-null   float64
 8   sentiment            1500 non-null   object 
 9   trust_score          1500 non-null   float64
dtypes: float64(4), int64(2), object(4)
memory usage: 117.3+ KB


None

From the `df.info()` output, we can see that there are 10 columns: 'text', 'topic', 'source_type', 'source_credibility', 'article_length', 'contradiction_score', 'semantic_similarity', 'readability_score', 'sentiment', and 'trust_score'.

'text', 'topic', 'source_type', and 'sentiment' are object type, while the others are numerical.

To perform logistic regression, we need a binary target variable. I will create a binary 'high_trust' variable from 'trust_score' (e.g., 1 if trust score is above the median, 0 otherwise). We also need to convert 'topic', 'source_type', and 'sentiment' into numerical representations using one-hot encoding.

In [3]:
median_trust_score = df['trust_score'].median()
df['high_trust'] = (df['trust_score'] > median_trust_score).astype(int)

print(f"Median trust score used for threshold: {median_trust_score:.2f}")
print("Distribution of 'high_trust' variable:")
print(df['high_trust'].value_counts())

display(df.head())

Median trust score used for threshold: 65.46
Distribution of 'high_trust' variable:
high_trust
1    750
0    750
Name: count, dtype: int64


,text,topic,source_type,source_credibility,article_length,contradiction_score,semantic_similarity,readability_score,sentiment,trust_score,high_trust
0,Politics related content discussing recent dev...,Politics,Government Website,8,3858,0.66,0.75,78.72,Neutral,74.54,1
1,Education related content discussing recent de...,Education,Research Paper,9,2895,0.87,0.56,89.87,Positive,83.81,1
2,Finance related content discussing recent deve...,Finance,Government Website,8,166,0.34,0.50,83.86,Positive,85.50,1
3,Finance related content discussing recent deve...,Finance,Government Website,8,4729,0.57,0.48,88.89,Positive,79.80,1
4,Healthcare related content discussing recent d...,Healthcare,News Article,6,2856,0.88,0.73,59.13,Neutral,54.79,0


In [4]:
categorical_features = ['topic', 'source_type', 'sentiment']

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

display(df_encoded.head())

print("Columns after one-hot encoding:")
print(df_encoded.columns.tolist())

,text,source_credibility,article_length,contradiction_score,semantic_similarity,readability_score,trust_score,high_trust,topic_Business,topic_Education,...,topic_Science,topic_Sports,topic_Technology,source_type_Educational Website,source_type_Government Website,source_type_News Article,source_type_Research Paper,source_type_Social Media Post,sentiment_Neutral,sentiment_Positive
0,Politics related content discussing recent dev...,8,3858,0.66,0.75,78.72,74.54,1,False,False,...,False,False,False,False,True,False,False,False,True,False
1,Education related content discussing recent de...,9,2895,0.87,0.56,89.87,83.81,1,False,True,...,False,False,False,False,False,False,True,False,False,True
2,Finance related content discussing recent deve...,8,166,0.34,0.50,83.86,85.50,1,False,False,...,False,False,False,False,True,False,False,False,False,True
3,Finance related content discussing recent deve...,8,4729,0.57,0.48,88.89,79.80,1,False,False,...,False,False,False,False,True,False,False,False,False,True
4,Healthcare related content discussing recent d...,6,2856,0.88,0.73,59.13,54.79,0,False,False,...,False,False,False,False,False,True,False,False,True,False


Columns after one-hot encoding:
['text', 'source_credibility', 'article_length', 'contradiction_score', 'semantic_similarity', 'readability_score', 'trust_score', 'high_trust', 'topic_Business', 'topic_Education', 'topic_Environment', 'topic_Finance', 'topic_Healthcare', 'topic_Politics', 'topic_Science', 'topic_Sports', 'topic_Technology', 'source_type_Educational Website', 'source_type_Government Website', 'source_type_News Article', 'source_type_Research Paper', 'source_type_Social Media Post', 'sentiment_Neutral', 'sentiment_Positive']


In [5]:
features_to_drop = ['text', 'trust_score', 'topic', 'source_type', 'sentiment']

columns_to_drop = [col for col in features_to_drop if col in df_encoded.columns]

X = df_encoded.drop(columns=columns_to_drop + ['high_trust'])
y = df_encoded['high_trust']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("First 5 rows of X:")
display(X.head())

Features (X) shape: (1500, 21)
Target (y) shape: (1500,)
First 5 rows of X:


,source_credibility,article_length,contradiction_score,semantic_similarity,readability_score,topic_Business,topic_Education,topic_Environment,topic_Finance,topic_Healthcare,...,topic_Science,topic_Sports,topic_Technology,source_type_Educational Website,source_type_Government Website,source_type_News Article,source_type_Research Paper,source_type_Social Media Post,sentiment_Neutral,sentiment_Positive
0,8,3858,0.66,0.75,78.72,False,False,False,False,False,...,False,False,False,False,True,False,False,False,True,False
1,9,2895,0.87,0.56,89.87,False,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
2,8,166,0.34,0.50,83.86,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
3,8,4729,0.57,0.48,88.89,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
4,6,2856,0.88,0.73,59.13,False,False,False,False,True,...,False,False,False,False,False,True,False,False,True,False


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1050, 21)
X_test shape: (450, 21)
y_train shape: (1050,)
y_test shape: (450,)


In [7]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [8]:
y_pred = model.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy Score: 0.9511111111111111

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.94      0.95       224
           1       0.94      0.96      0.95       226

    accuracy                           0.95       450
   macro avg       0.95      0.95      0.95       450
weighted avg       0.95      0.95      0.95       450

